# Notebook 18: Alternative Models Comparison
## FYP: Pre-Launch Game Reception Prediction (Steam + Google Play)

**Purpose:** a fair, equal-budget comparison of CatBoost against Logistic Regression,
Random Forest, XGBoost and LightGBM (plus optional TabNet), for the project proposal.

**Part A (Steam):** same data, label, 80/20 stratified split and 5-fold stratified CV as
`03_base_model_benchmark_v4`. Every model uses class weighting (no SMOTE). Each model is run
(a) with library default settings and (b) tuned with the same budget
(RandomizedSearchCV, 15 iterations, 5-fold CV on the training part only). The test set is
used once per final model. Bootstrap CIs and paired bootstrap differences on the test set.

**Part B (Google Play):** same data, label, 12 inputs and developer-grouped 5-fold CV as
`playstore_2_train_model`. Default and tuned (same budget) settings, scored inside the grouped CV.

**Inputs (read-only):** `data/processed/games_features_t4.csv`,
`outputs/results/03_winner_hyperparameters.json`, `notebooks/playstore_clean_dataset.csv`
**Outputs:** `outputs/results/18_*.csv` only. No existing notebook, data file or output is modified.

## 1. Setup, seeds and library versions

In [1]:
import sys, time, json, random, warnings, platform
from pathlib import Path
from importlib.metadata import version as _pkg_version

import numpy as np
import pandas as pd
import scipy, sklearn, xgboost, lightgbm, catboost
from scipy.stats import randint, loguniform

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.dummy           import DummyClassifier
from sklearn.preprocessing   import StandardScaler, OneHotEncoder
from sklearn.compose         import ColumnTransformer
from sklearn.pipeline        import Pipeline
from sklearn.model_selection import (StratifiedKFold, StratifiedGroupKFold, train_test_split,
                                     cross_validate, RandomizedSearchCV)
from sklearn.metrics         import f1_score, roc_auc_score, accuracy_score, make_scorer
from sklearn.base            import clone
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.config import FEATURES_T4_CSV, TARGET_COL, RANDOM_STATE, CV_FOLDS
assert RANDOM_STATE == SEED and CV_FOLDS == 5

RESULTS_DIR = PROJECT_ROOT / 'outputs' / 'results'
HP_JSON     = RESULTS_DIR / '03_winner_hyperparameters.json'
PLAY_CSV    = PROJECT_ROOT / 'notebooks' / 'playstore_clean_dataset.csv'
OUT_PREFIX  = '18_'

N_ITER = 15                      # RandomizedSearchCV budget, identical for every tuned model
NOTEBOOK_T0 = time.time()
RUNTIMES = {}

def save_csv(df, name):
    """All outputs of this notebook go to outputs/results/18_*.csv."""
    assert name.startswith(OUT_PREFIX)
    path = RESULTS_DIR / name
    df.to_csv(path, index=False)
    print(f'Saved: {path}')

VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
            'scipy': scipy.__version__, 'scikit-learn': sklearn.__version__,
            'xgboost': xgboost.__version__, 'lightgbm': lightgbm.__version__,
            'catboost': catboost.__version__}
try:
    import torch, pytorch_tabnet
    VERSIONS['torch'] = torch.__version__
    VERSIONS['pytorch-tabnet'] = _pkg_version('pytorch-tabnet')
    TABNET_AVAILABLE = True
except ImportError:
    TABNET_AVAILABLE = False

print(f'Project root : {PROJECT_ROOT}')
print(f'Platform     : {platform.platform()} | CPU threads: {__import__("os").cpu_count()}')
for k, v in VERSIONS.items():
    print(f'  {k:<15} {v}')

Project root : C:\Users\3214h\Documents\fyp-steam-reception
Platform     : Windows-11-10.0.26200-SP0 | CPU threads: 8
  python          3.13.9
  numpy           2.4.6
  pandas          3.0.3
  scipy           1.18.0
  scikit-learn    1.9.0
  xgboost         3.3.0
  lightgbm        4.7.0
  catboost        1.2.10
  torch           2.12.1+cpu
  pytorch-tabnet  4.1.0


# PART A: STEAM
## A1. Reproduce the notebook 03 data, split and CV

In [2]:
df = pd.read_csv(FEATURES_T4_CSV)
EXCLUDE_COLS = [TARGET_COL, 'short_description', 'dlc_count']   # same exclusions as notebook 03
T4_FEATURES = [c for c in df.columns if c not in EXCLUDE_COLS]

X = df[T4_FEATURES].values
y = df[TARGET_COL].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)

print(f'Input        : {FEATURES_T4_CSV}')
print(f'Rows         : {len(df):,}   features: {len(T4_FEATURES)}   positive rate: {y.mean():.3f}')
print(f'Train / test : {len(y_train):,} / {len(y_test):,}')
assert len(df) == 20_383 and len(T4_FEATURES) == 53
assert len(y_train) == 16_306 and len(y_test) == 4_077

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
SCORING = {'f1_macro': 'f1_macro',
           'f1_minority': make_scorer(f1_score, pos_label=0),   # label 0 = Not Well Received
           'roc_auc': 'roc_auc'}

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
SPW = neg / pos   # 'balanced' expressed as a positive-class weight (same as notebook 03)
print(f'Balanced positive-class weight (scale_pos_weight): {SPW:.4f}')

Input        : C:\Users\3214h\Documents\fyp-steam-reception\data\processed\games_features_t4.csv
Rows         : 20,383   features: 53   positive rate: 0.718
Train / test : 16,306 / 4,077
Balanced positive-class weight (scale_pos_weight): 0.3931


## A2. Shared evaluation helpers
CV metrics come from the training part only. `fit_and_test` refits on the full training part,
predicts the held-out test set once (threshold 0.5, the threshold notebook 03 selected), and
times training and prediction. Prediction time is reported two ways: per game inside one batch
call over the test set, and the median latency of a single-row call (what one app request costs).

In [3]:
TEST_PROBS = {}   # label -> test-set probabilities, used for the bootstrap

def cv_scores(est, X_, y_, cv_):
    t = time.perf_counter()
    s = cross_validate(est, X_, y_, cv=cv_, scoring=SCORING, n_jobs=1)
    return {'cv_macro_f1_mean': s['test_f1_macro'].mean(), 'cv_macro_f1_std': s['test_f1_macro'].std(),
            'cv_minority_f1_mean': s['test_f1_minority'].mean(),
            'cv_auc_mean': s['test_roc_auc'].mean(), 'cv_auc_std': s['test_roc_auc'].std(),
            'cv_time_s': time.perf_counter() - t}

def fit_and_test(label, est):
    m = clone(est)
    t = time.perf_counter(); m.fit(X_train, y_train); train_t = time.perf_counter() - t
    t = time.perf_counter(); p = m.predict_proba(X_test)[:, 1]; batch_t = time.perf_counter() - t
    m.predict_proba(X_test[:1])                      # warm-up
    lat = []
    for i in range(200):
        t = time.perf_counter(); m.predict_proba(X_test[i:i + 1]); lat.append(time.perf_counter() - t)
    pred = (p >= 0.5).astype(int)
    TEST_PROBS[label] = p
    return {'test_macro_f1': f1_score(y_test, pred, average='macro'),
            'test_minority_f1': f1_score(y_test, pred, pos_label=0),
            'test_auc': roc_auc_score(y_test, p),
            'test_accuracy': accuracy_score(y_test, pred),
            'train_time_s': train_t,
            'pred_ms_per_game_batch': batch_t / len(y_test) * 1000,
            'pred_ms_single_row': float(np.median(lat)) * 1000}

STEAM_ROWS = []
STEAM_COLS = ['model', 'setting', 'cv_macro_f1_mean', 'cv_macro_f1_std', 'cv_minority_f1_mean', 'cv_auc_mean',
              'test_macro_f1', 'test_minority_f1', 'test_auc', 'test_accuracy',
              'train_time_s', 'pred_ms_per_game_batch', 'pred_ms_single_row', 'cv_time_s', 'search_time_s', 'params']

def steam_table():
    return pd.DataFrame(STEAM_ROWS)[STEAM_COLS]

## A3. Reproduction check against notebook 03
Before comparing anything, confirm this notebook reproduces the notebook 03 numbers:
the notebook 03 Random Forest baseline (CV Macro F1 0.6677) and the tuned CatBoost from
`03_winner_hyperparameters.json` (CV Macro F1 0.6695, test Macro F1 0.6603).

In [4]:
hp = json.load(open(HP_JSON))
print(f'Loaded: {HP_JSON}')
print(json.dumps(hp['hyperparameters']))
CB_JSON_PARAMS = {k: v for k, v in hp['hyperparameters'].items() if k != 'random_seed'}

nb03_rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', max_depth=None,
                                 min_samples_leaf=2, random_state=SEED, n_jobs=-1)
cb_json = CatBoostClassifier(**CB_JSON_PARAMS, random_seed=SEED, verbose=0)

repro = []
r = cv_scores(nb03_rf, X_train, y_train, cv)
repro.append({'check': 'NB03 Random Forest baseline, CV Macro F1', 'expected': 0.6677, 'this_notebook': r['cv_macro_f1_mean']})
r_json = cv_scores(cb_json, X_train, y_train, cv)
repro.append({'check': 'NB03 tuned CatBoost (JSON), CV Macro F1', 'expected': hp['cv_macro_f1'], 'this_notebook': r_json['cv_macro_f1_mean']})
t_json = fit_and_test('CatBoost (NB03 JSON)', cb_json)
repro.append({'check': 'NB03 tuned CatBoost (JSON), test Macro F1', 'expected': hp['test_macro_f1_optimal'], 'this_notebook': t_json['test_macro_f1']})
repro.append({'check': 'NB03 tuned CatBoost (JSON), test AUC', 'expected': 0.7383, 'this_notebook': t_json['test_auc']})

repro_df = pd.DataFrame(repro)
repro_df['abs_diff'] = (repro_df['this_notebook'] - repro_df['expected']).abs()
repro_df['matches_4dp'] = repro_df['abs_diff'] < 0.00005
print(repro_df.to_string(index=False))
if not repro_df['matches_4dp'].all():
    print('\nWARNING: at least one number does not match notebook 03 to 4 decimals. Reported, not adjusted.')
save_csv(repro_df, '18_steam_reproduction_check.csv')

STEAM_ROWS.append({'model': 'CatBoost', 'setting': 'NB03 tuned (JSON)', **r_json, **t_json,
                   'search_time_s': np.nan, 'params': json.dumps(CB_JSON_PARAMS)})

Loaded: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\03_winner_hyperparameters.json
{"iterations": 1000, "learning_rate": 0.1, "depth": 4, "l2_leaf_reg": 5, "scale_pos_weight": 0.5, "random_seed": 42}


                                    check  expected  this_notebook  abs_diff  matches_4dp
 NB03 Random Forest baseline, CV Macro F1    0.6677         0.6677    0.0000         True
  NB03 tuned CatBoost (JSON), CV Macro F1    0.6695         0.6695    0.0000         True
NB03 tuned CatBoost (JSON), test Macro F1    0.6603         0.6603    0.0000         True
     NB03 tuned CatBoost (JSON), test AUC    0.7383         0.7383    0.0000         True
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_steam_reproduction_check.csv


## A4. (a) Default settings, class weighting only
Library defaults for every model, with only the class-imbalance setting changed
(`class_weight='balanced'`, `scale_pos_weight` = neg/pos, or `auto_class_weights='Balanced'`).
Logistic Regression is scaled and given `max_iter=1000` (as in notebook 03) so it converges.

In [5]:
def steam_defaults():
    return {
        'Logistic Regression': Pipeline([('scaler', StandardScaler()),
                                         ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED))]),
        'Random Forest': RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1),
        'XGBoost':       XGBClassifier(scale_pos_weight=SPW, random_state=SEED, n_jobs=-1, verbosity=0),
        'LightGBM':      LGBMClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1, verbose=-1),
        'CatBoost':      CatBoostClassifier(auto_class_weights='Balanced', random_seed=SEED, verbose=0),
    }

t0 = time.time()
DEFAULT_MODELS = steam_defaults()
for name, est in DEFAULT_MODELS.items():
    c = cv_scores(est, X_train, y_train, cv)
    t = fit_and_test(f'{name} (default)', est)
    STEAM_ROWS.append({'model': name, 'setting': 'default', **c, **t, 'search_time_s': np.nan, 'params': 'library defaults + class weighting'})
    print(f'{name:<20} CV Macro F1 {c["cv_macro_f1_mean"]:.4f} ± {c["cv_macro_f1_std"]:.4f} | '
          f'test Macro F1 {t["test_macro_f1"]:.4f} | train {t["train_time_s"]:.1f}s')
RUNTIMES['A4 Steam defaults'] = time.time() - t0
save_csv(steam_table(), '18_steam_model_comparison.csv')

Logistic Regression  CV Macro F1 0.6311 ± 0.0040 | test Macro F1 0.6343 | train 0.1s


Random Forest        CV Macro F1 0.6588 ± 0.0089 | test Macro F1 0.6573 | train 0.4s


XGBoost              CV Macro F1 0.6470 ± 0.0047 | test Macro F1 0.6493 | train 0.1s


LightGBM             CV Macro F1 0.6544 ± 0.0038 | test Macro F1 0.6436 | train 0.1s


CatBoost             CV Macro F1 0.6622 ± 0.0068 | test Macro F1 0.6573 | train 5.3s
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_steam_model_comparison.csv


## A5. (b) Tuned settings, same budget for every model
RandomizedSearchCV, `n_iter=15`, the same 5-fold stratified CV on the training part only,
Macro F1 as the refit target, `random_state=42`. Each search space covers capacity,
learning rate / regularisation, and the same grid of positive-class weights
(0.3, balanced = 0.393, 0.5, 0.7), so the notebook 03 CatBoost setting (scale_pos_weight 0.5)
is reachable. Logistic Regression gets the same 15-iteration budget over C and class weight.

Note: the CV score of the best of 15 candidates is optimistically biased (it was selected on
those same folds). The test score is the unbiased one.

In [6]:
W_POS = [0.3, SPW, 0.5, 0.7]
CW = [{0: 1.0, 1: w} for w in W_POS]
COMMON_LR = [0.01, 0.03, 0.05, 0.1]

SEARCHES = {
    'Logistic Regression': (
        Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=SEED))]),
        {'clf__C': loguniform(1e-3, 1e2), 'clf__class_weight': CW}),
    'Random Forest': (
        RandomForestClassifier(random_state=SEED, n_jobs=-1),
        {'n_estimators': randint(200, 800), 'max_depth': [None, 10, 20, 30],
         'min_samples_leaf': randint(1, 10), 'max_features': ['sqrt', 'log2', 0.3, 0.5], 'class_weight': CW}),
    'XGBoost': (
        XGBClassifier(random_state=SEED, n_jobs=-1, verbosity=0),
        {'n_estimators': [200, 500, 1000], 'learning_rate': COMMON_LR, 'max_depth': [3, 4, 6, 8],
         'subsample': [0.7, 0.85, 1.0], 'colsample_bytree': [0.7, 0.85, 1.0],
         'reg_lambda': [1, 3, 5, 10], 'scale_pos_weight': W_POS}),
    'LightGBM': (
        LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1, subsample_freq=1),
        {'n_estimators': [200, 500, 1000], 'learning_rate': COMMON_LR, 'num_leaves': [15, 31, 63],
         'max_depth': [-1, 4, 6, 8], 'min_child_samples': [10, 20, 50], 'subsample': [0.7, 0.85, 1.0],
         'colsample_bytree': [0.7, 0.85, 1.0], 'reg_lambda': [0, 1, 5, 10], 'class_weight': CW}),
    'CatBoost': (
        CatBoostClassifier(random_seed=SEED, verbose=0),
        {'iterations': [200, 500, 1000], 'learning_rate': COMMON_LR, 'depth': [4, 6, 8],
         'l2_leaf_reg': [1, 3, 5, 7, 9], 'scale_pos_weight': W_POS}),
}

def best_cv_from_search(rs):
    i, res = rs.best_index_, rs.cv_results_
    return {'cv_macro_f1_mean': res['mean_test_f1_macro'][i], 'cv_macro_f1_std': res['std_test_f1_macro'][i],
            'cv_minority_f1_mean': res['mean_test_f1_minority'][i],
            'cv_auc_mean': res['mean_test_roc_auc'][i], 'cv_auc_std': res['std_test_roc_auc'][i]}

def _fmt(v):
    if isinstance(v, (float, np.floating)): return round(float(v), 4)
    if isinstance(v, dict): return {int(k): round(float(w), 4) for k, w in v.items()}
    return v

t0 = time.time()
trials = []
TUNED_MODELS = {}
for name, (est, space) in SEARCHES.items():
    rs = RandomizedSearchCV(est, space, n_iter=N_ITER, scoring=SCORING, refit='f1_macro',
                            cv=cv, random_state=SEED, n_jobs=1)
    ts = time.perf_counter(); rs.fit(X_train, y_train); search_t = time.perf_counter() - ts
    best = clone(est).set_params(**rs.best_params_)
    TUNED_MODELS[name] = best
    c = best_cv_from_search(rs)
    t = fit_and_test(f'{name} (tuned)', best)
    params = {k: _fmt(v) for k, v in rs.best_params_.items()}
    STEAM_ROWS.append({'model': name, 'setting': f'tuned ({N_ITER}-iter random search)', **c, **t,
                       'cv_time_s': np.nan, 'search_time_s': search_t, 'params': json.dumps(params, default=str)})
    tr = pd.DataFrame(rs.cv_results_)[['params', 'mean_test_f1_macro', 'std_test_f1_macro', 'mean_test_roc_auc', 'rank_test_f1_macro']]
    tr.insert(0, 'model', name)
    trials.append(tr)
    print(f'{name:<20} best CV Macro F1 {c["cv_macro_f1_mean"]:.4f} ± {c["cv_macro_f1_std"]:.4f} | '
          f'test Macro F1 {t["test_macro_f1"]:.4f} | search {search_t:.0f}s | {params}')
    save_csv(steam_table(), '18_steam_model_comparison.csv')   # save after every model
RUNTIMES['A5 Steam tuning'] = time.time() - t0
trials_df = pd.concat(trials, ignore_index=True)
trials_df['params'] = trials_df['params'].astype(str)
save_csv(trials_df, '18_steam_search_trials.csv')

Logistic Regression  best CV Macro F1 0.6445 ± 0.0021 | test Macro F1 0.6450 | search 3s | {'clf__C': 0.4206, 'clf__class_weight': {0: 1.0, 1: 0.5}}
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_steam_model_comparison.csv


Random Forest        best CV Macro F1 0.6670 ± 0.0082 | test Macro F1 0.6711 | search 138s | {'class_weight': {0: 1.0, 1: 0.3931}, 'max_depth': 30, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'n_estimators': 545}
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_steam_model_comparison.csv


XGBoost              best CV Macro F1 0.6657 ± 0.0054 | test Macro F1 0.6542 | search 46s | {'subsample': 1.0, 'scale_pos_weight': 0.5, 'reg_lambda': 10, 'n_estimators': 1000, 'max_depth': 4, 'learning_rate': 0.03, 'colsample_bytree': 0.7}
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_steam_model_comparison.csv


LightGBM             best CV Macro F1 0.6667 ± 0.0038 | test Macro F1 0.6561 | search 30s | {'subsample': 1.0, 'reg_lambda': 5, 'num_leaves': 31, 'n_estimators': 200, 'min_child_samples': 10, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 1.0, 'class_weight': {0: 1.0, 1: 0.5}}
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_steam_model_comparison.csv


CatBoost             best CV Macro F1 0.6718 ± 0.0036 | test Macro F1 0.6686 | search 290s | {'scale_pos_weight': 0.5, 'learning_rate': 0.01, 'l2_leaf_reg': 9, 'iterations': 1000, 'depth': 8}
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_steam_model_comparison.csv
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_steam_search_trials.csv


## A6. Optional: TabNet (default settings)
Run only if `pytorch-tabnet` is installed and the 5-fold CV finishes within 15 minutes.
Inputs are standardised (fitted on each training fold). Class weighting is applied through a
weighted cross-entropy loss with balanced weights. Early stopping uses a 10% stratified split
carved out of each training fold, never the CV validation fold or the test set.
TabNet is run with default architecture only (no tuning), so it is not an equal-budget entry.

In [7]:
TABNET_LIMIT_S = 15 * 60

def tabnet_fit_predict(X_tr, y_tr, X_ev):
    from pytorch_tabnet.tab_model import TabNetClassifier
    torch.manual_seed(SEED)
    X_fit, X_es, y_fit, y_es = train_test_split(X_tr, y_tr, test_size=0.1, random_state=SEED, stratify=y_tr)
    sc = StandardScaler().fit(X_fit)
    w = torch.tensor(len(y_fit) / (2 * np.bincount(y_fit)), dtype=torch.float32)
    m = TabNetClassifier(seed=SEED, verbose=0, device_name='cpu')
    m.fit(sc.transform(X_fit), y_fit, eval_set=[(sc.transform(X_es), y_es)], eval_metric=['logloss'],
          max_epochs=100, patience=10, batch_size=1024, virtual_batch_size=128,
          loss_fn=torch.nn.CrossEntropyLoss(weight=w))
    return m, sc

TABNET_STATUS = 'not run'
if TABNET_AVAILABLE:
    t0 = time.time()
    f1s, mins, aucs = [], [], []
    for k, (tr, va) in enumerate(cv.split(X_train, y_train), 1):
        m, sc = tabnet_fit_predict(X_train[tr], y_train[tr], X_train[va])
        p = m.predict_proba(sc.transform(X_train[va]))[:, 1]; pred = (p >= 0.5).astype(int)
        f1s.append(f1_score(y_train[va], pred, average='macro')); mins.append(f1_score(y_train[va], pred, pos_label=0))
        aucs.append(roc_auc_score(y_train[va], p))
        print(f'  TabNet fold {k}: Macro F1 {f1s[-1]:.4f}  ({time.time() - t0:.0f}s elapsed)')
        if time.time() - t0 > TABNET_LIMIT_S:
            break
    cv_t = time.time() - t0
    if len(f1s) < CV_FOLDS:
        TABNET_STATUS = f'skipped: CV exceeded 15 minutes after {len(f1s)} folds'
    else:
        ts = time.perf_counter(); m, sc = tabnet_fit_predict(X_train, y_train, X_test); train_t = time.perf_counter() - ts
        ts = time.perf_counter(); p = m.predict_proba(sc.transform(X_test))[:, 1]; batch_t = time.perf_counter() - ts
        lat = []
        for i in range(200):
            ts = time.perf_counter(); m.predict_proba(sc.transform(X_test[i:i + 1])); lat.append(time.perf_counter() - ts)
        pred = (p >= 0.5).astype(int)
        TEST_PROBS['TabNet (default)'] = p
        STEAM_ROWS.append({'model': 'TabNet', 'setting': 'default (optional, untuned)',
                           'cv_macro_f1_mean': np.mean(f1s), 'cv_macro_f1_std': np.std(f1s),
                           'cv_minority_f1_mean': np.mean(mins), 'cv_auc_mean': np.mean(aucs), 'cv_auc_std': np.std(aucs),
                           'test_macro_f1': f1_score(y_test, pred, average='macro'),
                           'test_minority_f1': f1_score(y_test, pred, pos_label=0),
                           'test_auc': roc_auc_score(y_test, p), 'test_accuracy': accuracy_score(y_test, pred),
                           'train_time_s': train_t, 'pred_ms_per_game_batch': batch_t / len(y_test) * 1000,
                           'pred_ms_single_row': float(np.median(lat)) * 1000, 'cv_time_s': cv_t,
                           'search_time_s': np.nan, 'params': 'TabNet defaults, weighted CE, early stop on inner 10%'})
        TABNET_STATUS = f'ran: 5-fold CV took {cv_t / 60:.1f} min'
    RUNTIMES['A6 TabNet'] = time.time() - t0
    save_csv(steam_table(), '18_steam_model_comparison.csv')
else:
    TABNET_STATUS = 'skipped: pytorch-tabnet not installed'
print('TabNet:', TABNET_STATUS)


Early stopping occurred at epoch 23 with best_epoch = 13 and best_val_0_logloss = 0.6033


  TabNet fold 1: Macro F1 0.6227  (12s elapsed)



Early stopping occurred at epoch 27 with best_epoch = 17 and best_val_0_logloss = 0.60238


  TabNet fold 2: Macro F1 0.5957  (25s elapsed)



Early stopping occurred at epoch 33 with best_epoch = 23 and best_val_0_logloss = 0.60075


  TabNet fold 3: Macro F1 0.6139  (40s elapsed)


... (2 outputs left out)


Early stopping occurred at epoch 30 with best_epoch = 20 and best_val_0_logloss = 0.63052


  TabNet fold 5: Macro F1 0.6036  (75s elapsed)



Early stopping occurred at epoch 19 with best_epoch = 9 and best_val_0_logloss = 0.60966


Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_steam_model_comparison.csv
TabNet: ran: 5-fold CV took 1.2 min


## A7. Steam results table

In [8]:
steam_df = steam_table().sort_values(['setting', 'cv_macro_f1_mean'], ascending=[True, False]).reset_index(drop=True)
save_csv(steam_df, '18_steam_model_comparison.csv')
show = ['model', 'setting', 'cv_macro_f1_mean', 'cv_macro_f1_std', 'test_macro_f1', 'test_minority_f1',
        'test_auc', 'test_accuracy', 'train_time_s', 'pred_ms_per_game_batch', 'pred_ms_single_row']
print(steam_df[show].to_string(index=False))

Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_steam_model_comparison.csv
              model                       setting  cv_macro_f1_mean  cv_macro_f1_std  test_macro_f1  test_minority_f1  test_auc  test_accuracy  train_time_s  pred_ms_per_game_batch  pred_ms_single_row
           CatBoost             NB03 tuned (JSON)            0.6695           0.0063         0.6603            0.5214    0.7383         0.7172        3.6641                  0.0010              0.4101
           CatBoost                       default            0.6622           0.0068         0.6573            0.5362    0.7442         0.7000        5.2578                  0.0009              0.4121
      Random Forest                       default            0.6588           0.0089         0.6573            0.4924    0.7299         0.7366        0.4015                  0.0092             27.0491
           LightGBM                       default            0.6544           0.0038         0.643

## A8. Bootstrap 95% CIs and paired differences on the test set
1,000 resamples of the 4,077 test games (seed 42). The same resampled indices are used for
every model, so the differences are paired. A difference is clearly non-zero only if its 95% CI
excludes 0.

In [9]:
B = 1000
rng = np.random.default_rng(SEED)
BOOT_IDX = rng.integers(0, len(y_test), size=(B, len(y_test)))

def macro_f1_fast(y_true, y_pred):
    out = []
    for c in (0, 1):
        tp = np.sum((y_pred == c) & (y_true == c)); fp = np.sum((y_pred == c) & (y_true != c))
        fn = np.sum((y_pred != c) & (y_true == c))
        out.append(2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0)
    return (out[0] + out[1]) / 2

def boot_dist(label):
    pred = (TEST_PROBS[label] >= 0.5).astype(int)
    return np.array([macro_f1_fast(y_test[i], pred[i]) for i in BOOT_IDX])

BOOT_MODELS = ['CatBoost (NB03 JSON)', 'CatBoost (tuned)', 'CatBoost (default)',
               'LightGBM (tuned)', 'LightGBM (default)', 'Random Forest (tuned)', 'Random Forest (default)']
dists = {m: boot_dist(m) for m in BOOT_MODELS}
point = {m: f1_score(y_test, (TEST_PROBS[m] >= 0.5).astype(int), average='macro') for m in BOOT_MODELS}

boot_rows = []
for m in BOOT_MODELS:
    d = dists[m]
    boot_rows.append({'type': 'CI', 'comparison': m, 'point_estimate': point[m], 'boot_mean': d.mean(),
                      'boot_std': d.std(ddof=1), 'ci_low': np.percentile(d, 2.5), 'ci_high': np.percentile(d, 97.5),
                      'p_diff_le_0': np.nan})
PAIRS = [('CatBoost (NB03 JSON)', 'LightGBM (tuned)'), ('CatBoost (NB03 JSON)', 'Random Forest (tuned)'),
         ('CatBoost (tuned)', 'LightGBM (tuned)'), ('CatBoost (tuned)', 'Random Forest (tuned)'),
         ('CatBoost (default)', 'LightGBM (default)'), ('CatBoost (default)', 'Random Forest (default)')]
for a, b in PAIRS:
    d = dists[a] - dists[b]
    boot_rows.append({'type': 'paired diff', 'comparison': f'{a} minus {b}', 'point_estimate': point[a] - point[b],
                      'boot_mean': d.mean(), 'boot_std': d.std(ddof=1),
                      'ci_low': np.percentile(d, 2.5), 'ci_high': np.percentile(d, 97.5),
                      'p_diff_le_0': (d <= 0).mean()})
boot_df = pd.DataFrame(boot_rows)
boot_df['ci_excludes_0'] = np.where(boot_df['type'] == 'paired diff',
                                    (boot_df['ci_low'] > 0) | (boot_df['ci_high'] < 0), np.nan)
save_csv(boot_df, '18_steam_bootstrap.csv')
print(boot_df.to_string(index=False))

Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_steam_bootstrap.csv
       type                                       comparison  point_estimate  boot_mean  boot_std  ci_low  ci_high  p_diff_le_0  ci_excludes_0
         CI                             CatBoost (NB03 JSON)          0.6603     0.6603    0.0079  0.6455   0.6747          NaN            NaN
         CI                                 CatBoost (tuned)          0.6686     0.6683    0.0079  0.6529   0.6838          NaN            NaN
         CI                               CatBoost (default)          0.6573     0.6569    0.0079  0.6418   0.6724          NaN            NaN
         CI                                 LightGBM (tuned)          0.6561     0.6560    0.0079  0.6414   0.6713          NaN            NaN
         CI                               LightGBM (default)          0.6436     0.6435    0.0078  0.6273   0.6573          NaN            NaN
         CI                            Random Fores

## A9. Part A summary

In [10]:
BOOT_SD = boot_df.loc[boot_df['type'] == 'CI', 'boot_std'].mean()
tuned = steam_df[steam_df['setting'].str.startswith('tuned')].set_index('model')
default = steam_df[steam_df['setting'] == 'default'].set_index('model')
cb_json_row = steam_df[steam_df['setting'] == 'NB03 tuned (JSON)'].iloc[0]

def ranking(tab, col):
    s = tab[col].sort_values(ascending=False)
    return ', '.join(f'{m} {v:.4f}' for m, v in s.items())

STEAM_SUMMARY = []
STEAM_SUMMARY.append(f'Mean bootstrap SD of test Macro F1: {BOOT_SD:.4f}')
STEAM_SUMMARY.append(f'CV Macro F1, default : {ranking(default, "cv_macro_f1_mean")}')
STEAM_SUMMARY.append(f'CV Macro F1, tuned   : {ranking(tuned, "cv_macro_f1_mean")}')
STEAM_SUMMARY.append(f'Test Macro F1, tuned : {ranking(tuned, "test_macro_f1")}')
STEAM_SUMMARY.append(f'CatBoost NB03 JSON   : CV {cb_json_row.cv_macro_f1_mean:.4f}, test {cb_json_row.test_macro_f1:.4f}')
tree = tuned.drop(index=['Logistic Regression'])
gap_cv = tree['cv_macro_f1_mean'].max() - tree['cv_macro_f1_mean'].min()
STEAM_SUMMARY.append(f'Spread of tuned tree-model CV Macro F1: {gap_cv:.4f} '
                     f'({"smaller" if gap_cv < BOOT_SD else "larger"} than the bootstrap SD {BOOT_SD:.4f})')
for _, r in boot_df[boot_df['type'] == 'paired diff'].iterrows():
    STEAM_SUMMARY.append(f'{r.comparison}: {r.point_estimate:+.4f} (95% CI {r.ci_low:+.4f} to {r.ci_high:+.4f})'
                         f'{"  <- CI excludes 0" if r.ci_excludes_0 else ""}')
RUNTIMES['Part A total'] = time.time() - NOTEBOOK_T0
print('\n'.join(STEAM_SUMMARY))
print(f'\nPart A runtime: {RUNTIMES["Part A total"] / 60:.1f} min')

Mean bootstrap SD of test Macro F1: 0.0079
CV Macro F1, default : CatBoost 0.6622, Random Forest 0.6588, LightGBM 0.6544, XGBoost 0.6470, Logistic Regression 0.6311
CV Macro F1, tuned   : CatBoost 0.6718, Random Forest 0.6670, LightGBM 0.6667, XGBoost 0.6657, Logistic Regression 0.6445
Test Macro F1, tuned : Random Forest 0.6711, CatBoost 0.6686, LightGBM 0.6561, XGBoost 0.6542, Logistic Regression 0.6450
CatBoost NB03 JSON   : CV 0.6695, test 0.6603
Spread of tuned tree-model CV Macro F1: 0.0061 (smaller than the bootstrap SD 0.0079)
CatBoost (NB03 JSON) minus LightGBM (tuned): +0.0042 (95% CI -0.0058 to +0.0148)
CatBoost (NB03 JSON) minus Random Forest (tuned): -0.0108 (95% CI -0.0231 to +0.0017)
CatBoost (tuned) minus LightGBM (tuned): +0.0124 (95% CI +0.0036 to +0.0208)  <- CI excludes 0
CatBoost (tuned) minus Random Forest (tuned): -0.0025 (95% CI -0.0129 to +0.0081)
CatBoost (default) minus LightGBM (default): +0.0137 (95% CI +0.0032 to +0.0232)  <- CI excludes 0
CatBoost (defaul

# PART B: GOOGLE PLAY
Runs only after Part A has finished and saved its results.

## B1. Reproduce the `playstore_2_train_model` data, label, inputs and grouped CV
Same file (`playstore_clean_dataset.csv`, the no-suffix 102,215-row version, not `_all16`),
same 12 inputs, same label (average rating >= 4.0 with at least 50 ratings, already assigned by
notebook 1 and re-checked here), and the same developer-grouped CV. Note: `playstore_2_train_model`
uses `StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)` grouped by
`developer_hash`, not plain `GroupKFold`. It is reused exactly here so that no developer appears
in both a training and a validation fold. The fold list is computed once and shared by every model.

In [11]:
assert Path(STEAM_RESULTS_PATH := RESULTS_DIR / '18_steam_model_comparison.csv').exists(), 'Part A results not saved'
PART_B_T0 = time.time()

pdf = pd.read_csv(PLAY_CSV)
CAT = ['category', 'content_rating']
NUM = ['free', 'price_usd', 'ad_supported', 'in_app_purchases', 'size_mb', 'min_android',
       'has_website', 'has_privacy', 'size_was_missing', 'min_android_was_missing']
P_FEATURES = [f for f in CAT + NUM if f in pdf.columns]
CAT = [c for c in CAT if c in P_FEATURES]; NUM = [c for c in NUM if c in P_FEATURES]

print(f'Input      : {PLAY_CSV}')
print(f'Rows       : {len(pdf):,}   studios: {pdf["developer_hash"].nunique():,}   '
      f'categories: {pdf["category"].nunique()}   inputs: {len(P_FEATURES)}')
assert len(pdf) == 102_215 and len(P_FEATURES) == 12 and pdf['category'].nunique() == 14
assert pdf[P_FEATURES + ['label']].isna().sum().sum() == 0

relabel = ((pdf['audit_rating'] >= 4.0) & (pdf['audit_rating_count'] >= 50)).astype(int)
print(f'Label re-check (rating >= 4.0 and >= 50 ratings): {(relabel == pdf["label"]).mean() * 100:.2f}% agree; '
      f'well received {pdf["label"].mean() * 100:.1f}%')

XP = pdf[P_FEATURES].copy()
for c in CAT:
    XP[c] = XP[c].astype('category')
yp = pdf['label'].values
groups = pdf['developer_hash'].values
P_FOLDS = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(XP, yp, groups))
for k, (tr, va) in enumerate(P_FOLDS, 1):
    assert not set(groups[tr]) & set(groups[va])
print('Fold sizes (validation):', [len(va) for _, va in P_FOLDS], '| no developer shared across train/validation')
SPW_P = (yp == 0).sum() / (yp == 1).sum()
print(f'Balanced positive-class weight: {SPW_P:.4f}')

Input      : C:\Users\3214h\Documents\fyp-steam-reception\notebooks\playstore_clean_dataset.csv
Rows       : 102,215   studios: 38,334   categories: 14   inputs: 12
Label re-check (rating >= 4.0 and >= 50 ratings): 100.00% agree; well received 59.8%


Fold sizes (validation): [20443, 20445, 20443, 20442, 20442] | no developer shared across train/validation
Balanced positive-class weight: 0.6733


## B2. Models and encodings
- Logistic Regression: one-hot categories + standardised numeric inputs.
- Random Forest, XGBoost: one-hot categories, numeric inputs passed through.
- LightGBM: native categorical handling (pandas `category` dtype).
- CatBoost: native `cat_features`, as in `playstore_2_train_model` (passed as a tuple so sklearn can clone it).

All use class weighting. XGBoost's `scale_pos_weight` is computed from the whole label vector;
because the folds are stratified, each fold's ratio is the same to about 3 decimals.
Encoders sit inside each pipeline, so they are fitted on training folds only.

A reference row re-runs the exact `playstore_2_train_model` CatBoost (depth 4, lr 0.1,
500 iterations, l2 5, Balanced) to confirm its 0.6121 grouped-CV Macro F1 is reproduced.

In [12]:
def ohe(scale_num):
    return ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), CAT),
                              ('num', StandardScaler() if scale_num else 'passthrough', NUM)])

PLAY_DEFAULTS = {
    'Logistic Regression': Pipeline([('prep', ohe(True)),
                                     ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED))]),
    'Random Forest': Pipeline([('prep', ohe(False)),
                               ('clf', RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1))]),
    'XGBoost':  Pipeline([('prep', ohe(False)),
                          ('clf', XGBClassifier(scale_pos_weight=SPW_P, random_state=SEED, n_jobs=-1, verbosity=0))]),
    'LightGBM': LGBMClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1, verbose=-1),
    'CatBoost': CatBoostClassifier(auto_class_weights='Balanced', cat_features=tuple(CAT), random_seed=SEED, verbose=0),
}
PLAY_REFERENCE = CatBoostClassifier(depth=4, learning_rate=0.1, iterations=500, l2_leaf_reg=5,
                                    auto_class_weights='Balanced', cat_features=tuple(CAT),
                                    random_seed=SEED, verbose=0, thread_count=4)

PLAY_ROWS = []
PLAY_COLS = ['model', 'setting', 'cv_macro_f1_mean', 'cv_macro_f1_std', 'cv_auc_mean', 'cv_auc_std',
             'cv_minority_f1_mean', 'cv_time_s', 'search_time_s', 'params']

def play_table():
    return pd.DataFrame(PLAY_ROWS)[PLAY_COLS]

def play_cv(est):
    t = time.perf_counter()
    s = cross_validate(est, XP, yp, cv=P_FOLDS, scoring=SCORING, n_jobs=1)
    return {'cv_macro_f1_mean': s['test_f1_macro'].mean(), 'cv_macro_f1_std': s['test_f1_macro'].std(),
            'cv_minority_f1_mean': s['test_f1_minority'].mean(),
            'cv_auc_mean': s['test_roc_auc'].mean(), 'cv_auc_std': s['test_roc_auc'].std(),
            'cv_time_s': time.perf_counter() - t}

## B3. Majority baseline, reference CatBoost and default settings

In [13]:
t0 = time.time()
c = play_cv(DummyClassifier(strategy='most_frequent'))
PLAY_ROWS.append({'model': 'Majority baseline', 'setting': 'always predict majority', **c,
                  'search_time_s': np.nan, 'params': 'DummyClassifier(most_frequent)'})
print(f'{"Majority baseline":<20} Macro F1 {c["cv_macro_f1_mean"]:.4f} ± {c["cv_macro_f1_std"]:.4f} | AUC {c["cv_auc_mean"]:.4f}')

c = play_cv(PLAY_REFERENCE)
PLAY_ROWS.append({'model': 'CatBoost', 'setting': 'playstore_2 config (reference)', **c, 'search_time_s': np.nan,
                  'params': json.dumps({'depth': 4, 'learning_rate': 0.1, 'iterations': 500, 'l2_leaf_reg': 5,
                                        'auto_class_weights': 'Balanced'})})
print(f'{"CatBoost reference":<20} Macro F1 {c["cv_macro_f1_mean"]:.4f} ± {c["cv_macro_f1_std"]:.4f} | AUC {c["cv_auc_mean"]:.4f} '
      f'(playstore_2_train_model reported 0.6121 ± 0.0060, AUC 0.6690)')
PLAY_REF_MATCH = abs(c['cv_macro_f1_mean'] - 0.6121) < 0.00005
if not PLAY_REF_MATCH:
    print('WARNING: reference CatBoost does not match playstore_2_train_model to 4 decimals. Reported, not adjusted.')
save_csv(play_table(), '18_playstore_model_comparison.csv')

for name, est in PLAY_DEFAULTS.items():
    c = play_cv(est)
    PLAY_ROWS.append({'model': name, 'setting': 'default', **c, 'search_time_s': np.nan,
                      'params': 'library defaults + class weighting'})
    print(f'{name:<20} Macro F1 {c["cv_macro_f1_mean"]:.4f} ± {c["cv_macro_f1_std"]:.4f} | AUC {c["cv_auc_mean"]:.4f} | {c["cv_time_s"]:.0f}s')
    save_csv(play_table(), '18_playstore_model_comparison.csv')
RUNTIMES['B3 Play defaults'] = time.time() - t0

Majority baseline    Macro F1 0.3741 ± 0.0000 | AUC 0.5000


CatBoost reference   Macro F1 0.6121 ± 0.0053 | AUC 0.6690 (playstore_2_train_model reported 0.6121 ± 0.0060, AUC 0.6690)
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_playstore_model_comparison.csv


Logistic Regression  Macro F1 0.6029 ± 0.0079 | AUC 0.6516 | 1s
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_playstore_model_comparison.csv


Random Forest        Macro F1 0.5737 ± 0.0021 | AUC 0.6056 | 13s
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_playstore_model_comparison.csv


XGBoost              Macro F1 0.6122 ± 0.0029 | AUC 0.6667 | 2s
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_playstore_model_comparison.csv


LightGBM             Macro F1 0.6160 ± 0.0026 | AUC 0.6723 | 1s
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_playstore_model_comparison.csv


CatBoost             Macro F1 0.6155 ± 0.0030 | AUC 0.6707 | 315s
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_playstore_model_comparison.csv


## B4. Tuned settings, same budget, inside the grouped CV
RandomizedSearchCV with `n_iter=15` over the same shared grouped folds, Macro F1 as the target.
The search spaces mirror Part A; the class-weight grid is the balanced weight times
(0.75, 1.0, 1.25, 1.5). There is no separate test set here, so the reported tuned score is the
best of 15 candidates on the same folds and is optimistically biased by the selection. Treat
tuned-vs-default gains on Google Play as an upper bound.

In [14]:
W_POS_P = [round(SPW_P * m, 4) for m in (0.75, 1.0, 1.25, 1.5)]
CW_P = [{0: 1.0, 1: w} for w in W_POS_P]

PLAY_SEARCHES = {
    'Logistic Regression': (
        Pipeline([('prep', ohe(True)), ('clf', LogisticRegression(max_iter=1000, random_state=SEED))]),
        {'clf__C': loguniform(1e-3, 1e2), 'clf__class_weight': CW_P}),
    'Random Forest': (
        Pipeline([('prep', ohe(False)), ('clf', RandomForestClassifier(random_state=SEED, n_jobs=-1))]),
        {'clf__n_estimators': randint(200, 800), 'clf__max_depth': [None, 10, 20, 30],
         'clf__min_samples_leaf': randint(1, 10), 'clf__max_features': ['sqrt', 'log2', 0.3, 0.5],
         'clf__class_weight': CW_P}),
    'XGBoost': (
        Pipeline([('prep', ohe(False)), ('clf', XGBClassifier(random_state=SEED, n_jobs=-1, verbosity=0))]),
        {'clf__n_estimators': [200, 500, 1000], 'clf__learning_rate': COMMON_LR, 'clf__max_depth': [3, 4, 6, 8],
         'clf__subsample': [0.7, 0.85, 1.0], 'clf__colsample_bytree': [0.7, 0.85, 1.0],
         'clf__reg_lambda': [1, 3, 5, 10], 'clf__scale_pos_weight': W_POS_P}),
    'LightGBM': (
        LGBMClassifier(random_state=SEED, n_jobs=-1, verbose=-1, subsample_freq=1),
        {'n_estimators': [200, 500, 1000], 'learning_rate': COMMON_LR, 'num_leaves': [15, 31, 63],
         'max_depth': [-1, 4, 6, 8], 'min_child_samples': [10, 20, 50], 'subsample': [0.7, 0.85, 1.0],
         'colsample_bytree': [0.7, 0.85, 1.0], 'reg_lambda': [0, 1, 5, 10], 'class_weight': CW_P}),
    'CatBoost': (
        CatBoostClassifier(cat_features=tuple(CAT), random_seed=SEED, verbose=0),
        {'iterations': [200, 500, 1000], 'learning_rate': COMMON_LR, 'depth': [4, 6, 8],
         'l2_leaf_reg': [1, 3, 5, 7, 9], 'scale_pos_weight': W_POS_P}),
}

t0 = time.time()
p_trials = []
for name, (est, space) in PLAY_SEARCHES.items():
    rs = RandomizedSearchCV(est, space, n_iter=N_ITER, scoring=SCORING, refit=False,
                            cv=P_FOLDS, random_state=SEED, n_jobs=1)
    ts = time.perf_counter(); rs.fit(XP, yp); search_t = time.perf_counter() - ts
    res = rs.cv_results_
    i = int(np.argmin(res['rank_test_f1_macro']))
    params = {k: _fmt(v) for k, v in res['params'][i].items()}
    PLAY_ROWS.append({'model': name, 'setting': f'tuned ({N_ITER}-iter random search)',
                      'cv_macro_f1_mean': res['mean_test_f1_macro'][i], 'cv_macro_f1_std': res['std_test_f1_macro'][i],
                      'cv_auc_mean': res['mean_test_roc_auc'][i], 'cv_auc_std': res['std_test_roc_auc'][i],
                      'cv_minority_f1_mean': res['mean_test_f1_minority'][i], 'cv_time_s': np.nan,
                      'search_time_s': search_t, 'params': json.dumps(params, default=str)})
    tr = pd.DataFrame(res)[['params', 'mean_test_f1_macro', 'std_test_f1_macro', 'mean_test_roc_auc', 'rank_test_f1_macro']]
    tr.insert(0, 'model', name)
    p_trials.append(tr)
    print(f'{name:<20} best Macro F1 {res["mean_test_f1_macro"][i]:.4f} ± {res["std_test_f1_macro"][i]:.4f} | '
          f'AUC {res["mean_test_roc_auc"][i]:.4f} | search {search_t / 60:.1f} min | {params}')
    save_csv(play_table(), '18_playstore_model_comparison.csv')   # save after every model
RUNTIMES['B4 Play tuning'] = time.time() - t0
p_trials_df = pd.concat(p_trials, ignore_index=True)
p_trials_df['params'] = p_trials_df['params'].astype(str)
save_csv(p_trials_df, '18_playstore_search_trials.csv')

Logistic Regression  best Macro F1 0.6030 ± 0.0069 | AUC 0.6516 | search 0.3 min | {'clf__C': 0.9847, 'clf__class_weight': {0: 1.0, 1: 0.8417}}
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_playstore_model_comparison.csv


Random Forest        best Macro F1 0.6165 ± 0.0040 | AUC 0.6713 | search 12.1 min | {'clf__class_weight': {0: 1.0, 1: 0.6733}, 'clf__max_depth': None, 'clf__max_features': 'log2', 'clf__min_samples_leaf': 8, 'clf__n_estimators': 691}
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_playstore_model_comparison.csv


XGBoost              best Macro F1 0.6164 ± 0.0028 | AUC 0.6720 | search 2.2 min | {'clf__subsample': 1.0, 'clf__scale_pos_weight': 0.8417, 'clf__reg_lambda': 10, 'clf__n_estimators': 1000, 'clf__max_depth': 4, 'clf__learning_rate': 0.03, 'clf__colsample_bytree': 0.7}
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_playstore_model_comparison.csv


LightGBM             best Macro F1 0.6158 ± 0.0037 | AUC 0.6717 | search 1.2 min | {'subsample': 1.0, 'reg_lambda': 5, 'num_leaves': 31, 'n_estimators': 200, 'min_child_samples': 10, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 1.0, 'class_weight': {0: 1.0, 1: 0.8417}}
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_playstore_model_comparison.csv


CatBoost             best Macro F1 0.6165 ± 0.0047 | AUC 0.6689 | search 43.1 min | {'scale_pos_weight': 0.8417, 'learning_rate': 0.1, 'l2_leaf_reg': 3, 'iterations': 1000, 'depth': 6}
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_playstore_model_comparison.csv
Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_playstore_search_trials.csv


## B5. Google Play results table

In [15]:
play_df = play_table().sort_values(['setting', 'cv_macro_f1_mean'], ascending=[True, False]).reset_index(drop=True)
save_csv(play_df, '18_playstore_model_comparison.csv')
RUNTIMES['Part B total'] = time.time() - PART_B_T0
print(play_df[['model', 'setting', 'cv_macro_f1_mean', 'cv_macro_f1_std', 'cv_auc_mean', 'cv_auc_std',
               'cv_minority_f1_mean']].to_string(index=False))
print(f'\nPart B runtime: {RUNTIMES["Part B total"] / 60:.1f} min')

Saved: C:\Users\3214h\Documents\fyp-steam-reception\outputs\results\18_playstore_model_comparison.csv
              model                        setting  cv_macro_f1_mean  cv_macro_f1_std  cv_auc_mean  cv_auc_std  cv_minority_f1_mean
  Majority baseline        always predict majority            0.3741           0.0000       0.5000      0.0000               0.0000
           LightGBM                        default            0.6160           0.0026       0.6723      0.0036               0.5792
           CatBoost                        default            0.6155           0.0030       0.6707      0.0045               0.5775
            XGBoost                        default            0.6122           0.0029       0.6667      0.0036               0.5742
Logistic Regression                        default            0.6029           0.0079       0.6516      0.0083               0.5662
      Random Forest                        default            0.5737           0.0021       0.6056      0.

## 3. Plain-language summary (both platforms)

In [16]:
def top(tab, col='cv_macro_f1_mean'):
    r = tab.sort_values(col, ascending=False).iloc[0]
    return r['model'], r[col]

p_def = play_df[play_df['setting'] == 'default'].set_index('model')
p_tun = play_df[play_df['setting'].str.startswith('tuned')].set_index('model')
p_maj = play_df[play_df['model'] == 'Majority baseline'].iloc[0]

st_best_cv = top(steam_df[steam_df['model'] != 'TabNet'].reset_index())
st_best_test = top(tuned.reset_index(), 'test_macro_f1')
cb_vs = {r.comparison: r for _, r in boot_df[boot_df['type'] == 'paired diff'].iterrows()}
any_sig_win = any(r.ci_low > 0 for r in cb_vs.values())
any_sig_loss = any(r.ci_high < 0 for r in cb_vs.values())
cb_tuned_rank_st = int((tuned['cv_macro_f1_mean'] > tuned.loc['CatBoost', 'cv_macro_f1_mean']).sum()) + 1
p_tree_tuned = p_tun.drop(index=['Logistic Regression'])
cb_tuned_rank_p = int((p_tun['cv_macro_f1_mean'] > p_tun.loc['CatBoost', 'cv_macro_f1_mean']).sum()) + 1
p_spread = p_tree_tuned['cv_macro_f1_mean'].max() - p_tree_tuned['cv_macro_f1_mean'].min()
p_fold_sd = p_tree_tuned['cv_macro_f1_std'].mean()

lines = [
    'STEAM',
    f'- Highest CV Macro F1 overall: {st_best_cv[0]} ({st_best_cv[1]:.4f}). CatBoost ranks {cb_tuned_rank_st} of 5 among tuned models on CV.',
    f'- Highest test Macro F1 among tuned models: {st_best_test[0]} ({st_best_test[1]:.4f}). '
    f'The NB03 CatBoost scores {cb_json_row.test_macro_f1:.4f} on test.',
    f'- Tuned tree models are within {gap_cv:.4f} of each other on CV; the bootstrap SD of test Macro F1 is about {BOOT_SD:.4f}, '
    f'so the gaps are {"smaller than" if gap_cv < BOOT_SD else "comparable to or larger than"} the sampling noise.',
    f'- Paired bootstrap: {"at least one CatBoost advantage has a CI excluding 0" if any_sig_win else "no CatBoost advantage has a 95% CI excluding 0"}'
    f'{"; at least one CatBoost disadvantage has a CI excluding 0" if any_sig_loss else ""}.',
    '',
    'GOOGLE PLAY',
    f'- Majority baseline Macro F1 {p_maj.cv_macro_f1_mean:.4f}. Every model is well above it.',
    f'- Highest grouped-CV Macro F1, default: {top(p_def.reset_index())[0]} ({top(p_def.reset_index())[1]:.4f}); '
    f'tuned: {top(p_tun.reset_index())[0]} ({top(p_tun.reset_index())[1]:.4f}). CatBoost ranks {cb_tuned_rank_p} of 5 among tuned models.',
    f'- Tuned tree models span {p_spread:.4f}, against a typical fold SD of {p_fold_sd:.4f}.',
    '',
    'VERDICT',
]
if any_sig_win and cb_tuned_rank_st == 1 and cb_tuned_rank_p == 1:
    lines.append('- The evidence supports "CatBoost chosen over alternatives": it ranks first and at least one gap is outside the bootstrap noise.')
else:
    lines.append('- The evidence supports "CatBoost is comparable to the best alternatives", not "CatBoost is clearly better". '
                 'Differences between the gradient-boosting models and Random Forest are mostly within sampling noise. '
                 'Choosing CatBoost is defensible on other grounds (native categorical handling, stable folds, '
                 'SHAP support, fast inference) but not on accuracy alone.')
if any_sig_loss:
    lines.append('- Note: at least one alternative is significantly better than a CatBoost configuration on the Steam test set (see bootstrap table).')
RUNTIMES['Notebook total'] = time.time() - NOTEBOOK_T0
lines += ['', 'RUNTIME'] + [f'- {k}: {v / 60:.1f} min' for k, v in RUNTIMES.items()]
lines += ['', f'TabNet: {TABNET_STATUS}',
          f'Reproduction: NB03 checks all match = {bool(repro_df["matches_4dp"].all())}; '
          f'playstore_2 reference CatBoost matches = {PLAY_REF_MATCH}']
print('\n'.join(lines))

STEAM
- Highest CV Macro F1 overall: CatBoost (0.6718). CatBoost ranks 1 of 5 among tuned models on CV.
- Highest test Macro F1 among tuned models: Random Forest (0.6711). The NB03 CatBoost scores 0.6603 on test.
- Tuned tree models are within 0.0061 of each other on CV; the bootstrap SD of test Macro F1 is about 0.0079, so the gaps are smaller than the sampling noise.
- Paired bootstrap: at least one CatBoost advantage has a CI excluding 0.

GOOGLE PLAY
- Majority baseline Macro F1 0.3741. Every model is well above it.
- Highest grouped-CV Macro F1, default: LightGBM (0.6160); tuned: Random Forest (0.6165). CatBoost ranks 2 of 5 among tuned models.
- Tuned tree models span 0.0007, against a typical fold SD of 0.0038.

VERDICT
- The evidence supports "CatBoost is comparable to the best alternatives", not "CatBoost is clearly better". Differences between the gradient-boosting models and Random Forest are mostly within sampling noise. Choosing CatBoost is defensible on other grounds (nat